In [ ]:
# ============================================================
# DEMI Algorithm for Home Diagnosis of COVID-19
# Simplified graduate-level analysis
# ============================================================
#
# This script:
# 1. Loads the COVIDCARE patient data, DEMI knowledge base,
#    and survey dictionary.
# 2. Creates the PCR-confirmed analytic cohort.
# 3. Keeps information reasonably available at home.
# 4. Calculates DEMI pairwise odds ratios for PCR positivity.
# 5. Compares Logistic Regression, LASSO, and a boosting model.
# 6. Creates three figures for the final paper.
#
# Change FOLDER below to the folder that contains your 3 CSV files.
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    precision_score, recall_score, f1_score, roc_curve
)

# XGBoost is optional. If it is not installed, use sklearn boosting.
try:
    from xgboost import XGBClassifier
    USE_XGBOOST = True
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    USE_XGBOOST = False


# ============================================================
# 1. FILES AND SETTINGS
# ============================================================

FOLDER = Path(r"C:\YOUR\FOLDER\HERE")

RAW_FILE = FOLDER / "COVIDCARE_FORSUBMISSION_MIT_CLEANED_Phase_II_2021-12-03.csv"
KB_FILE = FOLDER / "COVIDCARE_DEMI_knowledgebase_v4.csv"
DICT_FILE = FOLDER / "COVIDCARE_survey_dictionary_v2_ForSubmission_MIT_Phase_II_2021-12-26.csv"

TARGET = "PCR Test Positive"
RANDOM_STATE = 42


# ============================================================
# 2. LOAD DATA
# ============================================================

df = pd.read_csv(RAW_FILE)
kb = pd.read_csv(KB_FILE)
dictionary = pd.read_csv(DICT_FILE)


In [ ]:
# Remove accidental spaces from column names
df.columns = df.columns.astype(str).str.strip()
kb.columns = kb.columns.astype(str).str.strip()
dictionary.columns = dictionary.columns.astype(str).str.strip()

print("Patient data:", df.shape)
print("Knowledge base:", kb.shape)
print("Dictionary:", dictionary.shape)

# Check that the correct patient file was loaded
if TARGET not in df.columns:
    print("\nPCR-related columns found:")
    print([c for c in df.columns if "PCR" in c.upper()])
    raise ValueError(
        f"'{TARGET}' was not found. Make sure RAW_FILE points to the "
        "COVIDCARE patient-level CSV, not the knowledge base or dictionary."
    )


# ============================================================
# 3. CREATE THE ANALYTIC COHORT
# ============================================================

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

# A known PCR result is required for supervised model evaluation
model_df = df.dropna(subset=[TARGET]).copy()
model_df[TARGET] = model_df[TARGET].astype(int)

print("\nAnalytic cohort:", model_df.shape[0])
print(model_df[TARGET].value_counts().rename(index={0: "PCR negative", 1: "PCR positive"}))


# ============================================================
# 4. SELECT INFORMATION AVAILABLE AT HOME
# ============================================================
#
# The dictionary is used to remove obvious study administration,
# identifiers, dates, and PCR-result leakage.
#
# This keeps the code understandable while remaining consistent
# with the project's five-tier idea.

dictionary["Variable Name"] = dictionary["Variable Name"].astype(str).str.strip()

description_map = {}
for _, row in dictionary.iterrows():
    variable = row["Variable Name"]
    description = str(row.get("Description", ""))
    prompt = str(row.get("Prompt", ""))
    description_map[variable] = (description + " " + prompt).lower()

exclude_words = [
    "pcr",
    "internal id",
    "submission date",
    "deid",
    "email",
    "phone",
    "shipping",
    "payment",
    "consent",
    "research study",
    "tested positive for covid-19 in the last 30 days"
]


In [ ]:
home_vars = []

for col in model_df.columns:
    if col == TARGET:
        continue

    text = (col + " " + description_map.get(col, "")).lower()

    # Remove obvious leakage / administrative variables
    if any(word in text for word in exclude_words):
        continue

    # Keep variables that have enough data to be useful
    if model_df[col].notna().sum() < 20:
        continue

    # Remove constant variables
    if model_df[col].nunique(dropna=True) < 2:
        continue

    home_vars.append(col)

print("\nHome-available predictors retained:", len(home_vars))


# ============================================================
# 5. FIVE-TIER SYSTEM
# ============================================================

def assign_tier(variable):
    """Assign a simple temporal tier based on variable description."""
    if variable == TARGET:
        return 4

    text = (variable + " " + description_map.get(variable, "")).lower()

    if any(x in text for x in ["at-home", "test strip", "pink line", "blue line"]):
        return 3

    if any(x in text for x in ["vaccine", "vaccination", "vaccinated", "flu shot"]):
        return 1

    if any(x in text for x in ["age", "gender", "race", "ethnicity", "birth sex"]):
        return 0

    return 2


tier_table = pd.DataFrame({
    "Variable": home_vars,
    "Tier": [assign_tier(v) for v in home_vars]
})

print("\nTier counts:")
print(tier_table["Tier"].value_counts().sort_index())


# ============================================================
# 6. DEMI PAIRWISE ASSOCIATIONS WITH PCR POSITIVITY
# ============================================================
#
# The knowledge base contains 2x2 counts for each concept pair.
# We calculate an odds ratio and log odds ratio for concepts
# whose target is PCR positivity.

In [ ]:

pcr_kb = kb[kb["target_concept_code"] == "target_pcr_positive"].copy()

count_cols = [
    "n_code_target",
    "n_code_no_target",
    "n_target_no_code",
    "n_no_code_no_target"
]

for col in count_cols:
    pcr_kb[col] = pd.to_numeric(pcr_kb[col], errors="coerce").fillna(0)

# Haldane-Anscombe 0.5 correction prevents division by zero
pcr_kb["odds_ratio"] = (
    (pcr_kb["n_code_target"] + 0.5) *
    (pcr_kb["n_no_code_no_target"] + 0.5)
) / (
    (pcr_kb["n_code_no_target"] + 0.5) *
    (pcr_kb["n_target_no_code"] + 0.5)
)

pcr_kb["log_odds_ratio"] = np.log(pcr_kb["odds_ratio"])
pcr_kb["total_with_code"] = (
    pcr_kb["n_code_target"] + pcr_kb["n_code_no_target"]
)

# Avoid displaying extremely unstable effects based on tiny counts
demi_results = pcr_kb[pcr_kb["total_with_code"] >= 10].copy()
demi_results = demi_results.sort_values(
    "log_odds_ratio", key=lambda x: x.abs(), ascending=False
)

print("\nTop DEMI pairwise associations:")
print(
    demi_results[
        ["concept_code", "odds_ratio", "log_odds_ratio"]
    ].head(10).to_string(index=False)
)

demi_results.to_csv(FOLDER / "DEMI_pairwise_results.csv", index=False)


# ============================================================
# 7. PREPARE DATA FOR MACHINE LEARNING
# ============================================================

X = model_df[home_vars].copy()
y = model_df[TARGET].copy()

categorical_cols = []
numeric_cols = []

for col in home_vars:
    if pd.api.types.is_numeric_dtype(X[col]) and X[col].nunique(dropna=True) > 10:
        numeric_cols.append(col)
    else:
        categorical_cols.append(col)

preprocess = ColumnTransformer([
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_cols
    ),
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]),
        numeric_cols
    )

In [ ]:
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE
)


# ============================================================
# 8. DEFINE THREE MODELS
# ============================================================

logistic = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        solver="liblinear"
    ))
])

lasso = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegressionCV(
        cv=5,
        penalty="l1",
        solver="saga",
        scoring="roc_auc",
        class_weight="balanced",
        max_iter=5000,
        random_state=RANDOM_STATE
    ))
])

if USE_XGBOOST:
    negative = (y_train == 0).sum()
    positive = (y_train == 1).sum()

    boosting = Pipeline([
        ("prep", preprocess),
        ("model", XGBClassifier(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            scale_pos_weight=negative / positive
        ))
    ])
    boosting_name = "XGBoost"

else:
    boosting = Pipeline([
        ("prep", preprocess),
        ("model", GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_STATE
        ))
    ])
    boosting_name = "Gradient Boosting"


models = {
    "Logistic Regression": logistic,
    "LASSO": lasso,
    boosting_name: boosting
}


# ============================================================
# 9. TRAIN AND EVALUATE MODELS
# ============================================================

results = []
probabilities = {}
fitted_models = {}

In [ ]:

for name, model in models.items():

    print("\nTraining:", name)

    model.fit(X_train, y_train)
    fitted_models[name] = model

    probability = model.predict_proba(X_test)[:, 1]
    prediction = (probability >= 0.50).astype(int)

    probabilities[name] = probability

    tn, fp, fn, tp = confusion_matrix(
        y_test, prediction, labels=[0, 1]
    ).ravel()

    results.append({
        "Model": name,
        "AUC": roc_auc_score(y_test, probability),
        "Accuracy": accuracy_score(y_test, prediction),
        "Sensitivity": recall_score(y_test, prediction),
        "Specificity": tn / (tn + fp),
        "Precision": precision_score(y_test, prediction, zero_division=0),
        "F1": f1_score(y_test, prediction)
    })


results_df = pd.DataFrame(results).sort_values("AUC", ascending=False)

print("\nMODEL PERFORMANCE")
print(results_df.round(3).to_string(index=False))

results_df.to_csv(FOLDER / "model_performance.csv", index=False)


# ============================================================
# 10. FIGURE 1: FIVE-TIER DIAGNOSTIC DAG
# ============================================================

G = nx.DiGraph()

G.add_edges_from([
    ("Demographics", "Symptoms / Exposures"),
    ("Vaccination", "Symptoms / Exposures"),
    ("Demographics", "PCR Positive"),
    ("Vaccination", "PCR Positive"),
    ("Symptoms / Exposures", "At-home Test"),
    ("Symptoms / Exposures", "PCR Positive"),
    ("At-home Test", "PCR Positive")
])

plt.figure(figsize=(10, 6))

positions = {
    "Demographics": (0, 2),
    "Vaccination": (0, 1),
    "Symptoms / Exposures": (1, 1.5),
    "At-home Test": (2, 1.5),
    "PCR Positive": (3, 1.5)
}

nx.draw(
    G,
    positions,
    with_labels=True,
    node_size=3500,
    font_size=10,
    arrows=True,
    arrowsize=20
)

plt.title("Five-Tier COVID-19 Diagnostic Network")
plt.axis("off")
plt.tight_layout()
plt.savefig(FOLDER / "Figure1_DAG.png", dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 11. FIGURE 2: ROC CURVES
# ============================================================

plt.figure(figsize=(8, 6))

for name, probability in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, probability)
    auc = roc_auc_score(y_test, probability)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for COVID-19 Prediction Models")
plt.legend()
plt.tight_layout()
plt.savefig(FOLDER / "Figure2_ROC.png", dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 12. FIGURE 3: TOP DEMI LOG ODDS RATIOS
# ============================================================

top_effects = demi_results.head(10).copy()
top_effects = top_effects.sort_values("log_odds_ratio")

plt.figure(figsize=(9, 6))
plt.barh(top_effects["concept_code"], top_effects["log_odds_ratio"])
plt.axvline(0, linewidth=1)
plt.xlabel("Log Odds Ratio")
plt.ylabel("DEMI Concept")
plt.title("Top Pairwise DEMI Associations With PCR Positivity")
plt.tight_layout()
plt.savefig(FOLDER / "Figure3_DEMI_Effects.png", dpi=300, bbox_inches="tight")
plt.show()


# ============================================================
# 13. SIMPLE PATIENT-LEVEL PREDICTION
# ============================================================

best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]

def predict_case(patient_information):
    """
    Estimate probability of PCR-positive COVID-19.

    Example:
        patient = {
            "AGE_VARIABLE_NAME": 30,
            "FEVER_VARIABLE_NAME": 1
        }
        predict_case(patient)
    """

    patient = pd.DataFrame(
        [{col: np.nan for col in home_vars}]
    )

    for variable, value in patient_information.items():
        if variable in patient.columns:
            patient.loc[0, variable] = value

    probability = best_model.predict_proba(patient)[0, 1]
    return probability


print("\nAnalysis complete.")
print("Best model by AUC:", best_model_name)
print("Results and figures were saved to:", FOLDER)